#### FPVS pypeline 2 pynt 0
This is the second edition of the FPVS pypeline where all processes/steps in the pipeline are functionalized so that they can be completed in a more streamlined manner in addition to this application being distributed as a package. 
steps 

1. Preprocess step 1

## Import libraries

In [2]:
%reset -f
import cust_funcs as cf
import FPVS_pycage as FPVS
from pathlib import Path
import importlib
# importlib.reload(FPVS)

## Preprocessing phase 1 
0. Remodelling the lw6 and mat data to make them more Python-friendly
1. Rename channels
2. Changing the channel locations and adding spherical coordinates to the meta (lw6) data
3. Downsampling the mat_data 
4. Deleting the EXG X and status channels
5. Bandpass and notch filtering

In [ ]:
importlib.reload(cf)
importlib.reload(FPVS)
matfilepath = Path("D:\Goffaux lab\Data set\Python_CSVfiles\CHMA0501.mat")
lw6filepath = matfilepath.with_suffix(".lw6")
subjid = matfilepath.stem
mat_data, meta_data , metafilepath = FPVS.preprocessFPVSdata_phase1(matfilepath,lw6filepath)

In [31]:
## test code
import numpy as np
from pathlib import Path
metafilepath = Path("D:\Goffaux lab\Data set\Python_CSVfiles\CHMA0501 mdata.npy")
# meta_data = cf.openMetadata(metafilepath)
meta_data = np.load(metafilepath,allow_pickle=True)
# print(meta_data.keys)

transformed = meta_data.item()
print(transformed.keys())
# print(meta_data['chanlocs'])

dict_keys(['filetype', 'name', 'tags', 'history', 'origins', 'datasize', 'xstart', 'ystart', 'zstart', 'xstep', 'ystep', 'zstep', 'chanlocs', 'events', 'fs', 'activation', 'fields'])


In [26]:
import scipy.io as sci
del meta_data,meta_data1
metafilepath1 = Path("D:\Goffaux lab\Data set\Python_CSVfiles\CHMA0501.lw6")
meta_data = cf.rebrand_lw6data(metafilepath1)
print(meta_data)

print('-'*30)

meta_data1 = cf.assign(meta_data)
print(meta_data1)


the keys in old data are:  ('filetype', 'name', 'tags', 'history', 'datasize', 'xstart', 'ystart', 'zstart', 'xstep', 'ystep', 'zstep', 'chanlocs', 'events')
the keys in the new data are:  dict_keys(['filetype', 'name', 'tags', 'history', 'origins', 'datasize', 'xstart', 'ystart', 'zstart', 'xstep', 'ystep', 'zstep', 'chanlocs', 'events', 'fs', 'activation', 'fields'])
{'filetype': array(array(['time_amplitude'], dtype='<U14'), dtype=object), 'name': array(array(['CHMA0501'], dtype='<U8'), dtype=object), 'tags': {}, 'history': {}, 'origins': {'gui_info': ['no gui, no info'], 'originalfilepath': WindowsPath('D:/Goffaux lab/Data set/Python_CSVfiles/CHMA0501.lw6')}, 'datasize': array(array([[      1,      73,       1,       1,       1, 7374848]],
      dtype=int32), dtype=object), 'xstart': array(array([[-0.]]), dtype=object), 'ystart': array(array([[0]], dtype=uint8), dtype=object), 'zstart': array(array([[0]], dtype=uint8), dtype=object), 'xstep': array(array([[0.00048828]]), dtype=obje

In [31]:
print(meta_data1)

AttributeError: 'dict' object has no attribute 'dtype'

In [3]:
import numpy as np
from pathlib import Path



meta_data = cf.assign(meta_data)
print(meta_data['chanlocs']['labels'])




['Fp1', 'AF7', 'AF3', 'F1', 'F3', 'F5', 'F7', 'FT7', 'FC5', 'FC3', 'FC1', 'C1', 'C3', 'C5', 'T7', 'TP7', 'CP5', 'CP3', 'CP1', 'P1', 'P3', 'P5', 'P7', 'P9', 'PO7', 'PO3', 'O1', 'Iz', 'Oz', 'POz', 'Pz', 'CPz', 'Fpz', 'Fp2', 'AF8', 'AF4', 'AFz', 'Fz', 'F2', 'F4', 'F6', 'F8', 'FT8', 'FC6', 'FC4', 'FC2', 'FCz', 'Cz', 'C2', 'C4', 'C6', 'T8', 'TP8', 'CP6', 'CP4', 'CP2', 'P2', 'P4', 'P6', 'P8', 'P10', 'PO8', 'PO4', 'O2', 'EXG1', 'EXG2', 'EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8', 'Status']


In [6]:
print(meta_data.keys())

dict_keys(['filetype', 'name', 'tags', 'history', 'origins', 'datasize', 'xstart', 'ystart', 'zstart', 'xstep', 'ystep', 'zstep', 'chanlocs', 'events', 'fs', 'activation', 'fields'])


In [ ]:
print(meta_data.keys())
newmetafilepath = metafilepath.with_name(metafilepath.stem + "trial run")
newmetafilepath = newmetafilepath.with_suffix(".npy")

np.save(newmetafilepath,meta_data)
print("data_saved")

In [ ]:
new_meta_data = np.load(newmetafilepath,allow_pickle=True)
print(new_meta_data)

## Performing ICA 
Because jupyter is asynchronous, i have to split the perform_ICA, overlay ICA and apply ICA separately, in three different cells. 

In [ ]:
importlib.reload(FPVS)
matfilepath = metafilepath.with_name(metafilepath.stem.removesuffix(" mdata") + metafilepath.suffix)

print(metafilepath.exists())
print(metafilepath)
print(matfilepath)
ica, raw, ica_data, idx = FPVS.preprocessFPVSdata_performICA(matfilepath=matfilepath, metafilepath = metafilepath)

## Overlay ICA data on actual activation

In [ ]:
import numpy as np
# %matplotlib widgets
importlib.reload(cf)
mat_data = np.load(matfilepath)
meta_data = np.load(metafilepath, allow_pickle=True)
labels = np.squeeze(meta_data["chanlocs"][0][0]["labels"])
# idx = np.where(
cf.showmeICAoverlayedondata(mat_data, ica_data,labels= labels)

## Apply ICA to the actual data. 

In [ ]:
## rmidx is the variable where the Independent components with blink are stored and will be removed in the next cell
# rmidx = [0,2,3,7,8,9,10,12] #ADVI
rmidx = [0,2,3,5,6,9,11,13,15,16,17,18,19,20,21,23,24] # 24 looks dicey CHMA


In [ ]:
importlib.reload(cf)
mat_data, meta_data, metafilepath = FPVS.preprocessFPVSdata_applyICA(matfilepath, metafilepath, ica , raw, rmidx)

## Segmenting the data 

In [ ]:
importlib.reload(cf)
print(mat_data.shape)
matfilepath = metafilepath.with_name(metafilepath.stem.removesuffix(" mdata") + metafilepath.suffix)
mat_data, meta_data, metafilepath = FPVS.preprocesssFPVSdata_segmentation(matfilepath, metafilepath)

## Interpolate channels

In [ ]:
matfilepath = metafilepath.with_name(metafilepath.stem.removesuffix(" mdata") + metafilepath.suffix)
mat_data = np.load(matfilepath)
meta_data = np.load(metafilepath ,allow_pickle=True)
cf.showmeSignalUI(mat_data, meta_data,ylim=150)

In [ ]:
interp_chnames = ['P1','P3','PO3']
bad_chids = ['Fp1','CP6','I2']
print(labels)

In [ ]:
importlib.reload(FPVS)
mat_data,meta_data,metafile = FPVS.preprocessFPVSdata_phase2(matfilepath,metafilepath,interp_chnames=interp_chnames,bad_but_ignore=bad_chids)

## Post processing 
1. FFT transform
2. averaging within trials
3. Frequency window chunking
4. Chunk selection( and splitting based on baseline and oddball chunks)
5. Sum of chunks/harmonics
6. baseline correction (I havent done SNR correction yet)

In [ ]:
odd_data, bl_data, oddfilepath, blfilepath = FPVS.postprocessFPVSdata("200", metafilepath.parent)